# Project 3: Checkpoint 2

This part of the project has you: 
* Creating a train & test set 
* Constructing a regression pipeline (Selecting between linear regression and k-nn regression) 
* Performing cross-validation to either (a) select parameters (knn regression) or (b) evaluate the quality of predictors (linear regression) 
* Evaluating your model's performance on the test set. 

## Grading Criteria 

All project checkpoints are graded on the following criteria. **You must have satisfactory completeness** when this assignment is **first due**. You will know you have satisfactory completeness by having code in every code cell that attempts to answer the question and runs, and clearly labeled text cells that meaningfully attempts to answer every question in complete sentences. 

<details>

| **Criteria**            | **Excellent** | **Satisfactory** | **Revise** | **Not Assessable** |
|-------------------------|---------------|------------------|------------|---------------------|
| **Completeness**        | All required components are fully completed. The work goes beyond minimum requirements, with attention to detail and clear structure. | Most components are completed as required. Minor elements may be missing or incomplete, but the core work is present and functional. | Several key parts are missing, incomplete, or incorrect. Some progress is made, but the submission lacks important elements. | Very little or none of the required work is present. Submission is incomplete or off-topic. |
| **Justifying Choices**  | Clear and thoughtful justification is given for all major decisions. Demonstrates awareness of alternatives and logical reasoning. | Reasoning is provided for most decisions. Justifications are basic but show effort to explain choices. | Reasoning is limited, unclear, or superficial. Many decisions are unexplained or poorly justified. | No meaningful justification is provided. Decisions appear arbitrary or are copied without explanation. |
| **Coding Practices**    | Code is correct, runs without errors, and clearly applies methods or techniques taught in class. The code is well-organized, uses meaningful variable names, and is easy to follow. Comments enhance clarity. | Code mostly works and shows a clear attempt to use appropriate class-taught methods. There may be minor errors or inconsistencies, but the logic is mostly sound. Code is readable, though may lack polish or full clarity. | Code has significant issues (e.g., some of it does not run, gives incorrect output, or follows flawed logic). May not use methods from class or apply them incorrectly. Readability is poor or confusing, with unclear structure or variable names. | Code is substantially missing, does not run, or shows no understanding of class-taught concepts. May be copied, off-topic, or nonsensical. |
| **Visualization Quality (if applicable)** | Visualizations are clear, appropriate, well-labeled, and enhance understanding. Thoughtful choices of chart types and formatting. | Visualizations are generally appropriate and correctly formatted. Some labels or design choices could be improved. | Visualizations are present but may be unclear, mislabeled, or poorly chosen. Minimal effort to align visuals with data meaning. | Visualizations are missing, incorrect, or irrelevant to the task. |
| **Accurate Interpretation** | Interpretations are accurate, well-supported, and clearly communicated. Student demonstrates strong understanding of results. | Interpretations are mostly accurate and reasonable. May lack depth or precision, but show basic understanding. | Interpretations contain misunderstandings or are too vague. Some conclusions may not align with the data. | No interpretation is provided, or interpretations are clearly incorrect or misleading. |
    
</details>


### 1. Load data, Plan Ahead

a) Create a code cell, import `pandas` with the alias `pd`, then copy your saved dataframe from last week  (with any new columns) to the "data" folder and load it with a sensible name. [Coding Practices]

b) In a text cell below, recall your plans: For this regression problem, what do you plan to use as your predictors, and what do you plan to predict? **Make sure that you have ensured your set of predictors and response variables was approved by Prof. Masden - if not, you might not get full credit here and will have difficulty proceeding. For instance, you will need at least two predictors** [Completeness]
 
c) Look ahead at the instructions for the next several parts. Based on your previous visualizations of the relationship between your predictors and your response variable, do you plan to use k-nn regression or linear regression? If so, what features of the data do you think make this the better choice? Finally, what computations do you plan to do? [Justification]

In [3]:
import pandas as pd 
credit_clean = pd.read_csv("/home/user/Project_Checkpoints/Project_3_Checkpoint_1/credit_clean.csv")

For this regression problem my question would be can we predict a person's credit limit, and I will be using age, education, average monthly bill, and average monthly payment as my predictor. 


I plan to use k-nn regression. Looking at my scatter matrix from last week, the relationship between my predictors and credit limit were spread out and did not follow a straight line. Average bill and average payment had a fan shape with a few very large outliers, and age showed almost no pattern. Linear regression assumes a straight line relationship, so K-nn would be a better option. 

So for knn regression, I will first split my data into training set and test set, standardise my numerical columns, one-hot encode Education, and build a knn regression pipeline. I will then fit my pipeline on the subtraining se to see how the model does. I will then compute the RMSPE and see how the model performs. I will visulaize my model's prediction. Later I will also use cross validation to try different values of kk and choose the one with the lowest RMSPE. 


In [18]:
import plotly.express as px
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.compose import make_column_transformer
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsRegresso
# Output dataframes instead of arrays
set_config(transform_output="pandas")

## Regression 

### 2. Getting Set up (Making your Pipeline)

1. Construct a training dataframe and a test dataframe from your saved dataframe. Save them as `yourname_train.csv` and `yourname_test.csv` (or similar). In all future analyses, you should load *this* train-test split. 

2. Construct your preprocessor steps. You might choose to include: 
   * Standardization (This is no longer ALWAYS necessary - consider whether you're doing linear regression vs. k-nn regression)
   * *One-hot encoding*  (Remember there is a difference between doing this for knn and linear regression)
   * *Oversampling*
   * Creating an *imputer*
   
3. Create your regressor model specification (either k-nn or linear regression)

3. Justification: Why do you expect the preprocessing steps you performed to improve your regression model?  If you did not choose to use any preprocessing steps, explain why you do not think they are necessary. Basically, explain why you did or did not choose *each* of the preprocessors above. 

4. Construct a pipeline with all appropriate pre-processing steps, ending with your regression model. 

In [4]:
credit_train, credit_test = train_test_split(credit_clean,
                                            train_size = 0.75,
                                            random_state = 42)
credit_train.to_csv("zachary_train.csv", index = False)
credit_test.to_csv("zachary_test.csv", index = False)

In [15]:
numeric_col = ["Age", "avg_bill", "avg_payment"]
category_col = ["Education"]

preprocessor = make_column_transformer(
(StandardScaler(), numeric_col),
(OneHotEncoder(sparse_output = False), ["Education"]))

In [19]:
knn_spec = KNeighborsRegressor(n_neighbors = 20)

I do standardize age, average bill and average payment so they are on the same scale and can be compared. Then I did one hot encoding, one hot encoded Educatoin because the number is 1 to 4 which are category. 


In [20]:
credit_pipe = make_pipeline(preprocessor, knn_spec)
credit_pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columntransformer', ...), ('kneighborsregressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('standardscaler', ...), ('onehotencoder', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sp

### 3. Check your code: Perform a fit and predict on one subtraining-validation set

1. Check that your pipeline's `.fit()` method works by fitting it on a subtraining set
2. Check that your pipelines `.predict()` method works by predicting on a validation set
3. Obtain the RMSPE on your training set. What does this number mean, in context? 

(At this point, if you can't get this to work -- turn this in anyway; I'll help you figure out what's going wrong. This is better than waiting to turn it in later)

In [24]:
credit_subtrain, credit_val = train_test_split(credit_train, train_size = 0.8, random_state = 42)
col = ["Age", "Education", "avg_bill", "avg_payment"]

X = credit_subtrain[col]
y = credit_subtrain["Credit Limit"]

fit_credit_pipe = credit_pipe.fit(X, y)

In [32]:
credit_val["predicted"] = fit_credit_pipe.predict(credit_val[col])
credit_val[["Credit Limit", "predicted"]]

,Credit Limit,predicted
6423,180000,242500.0
5058,200000,209500.0
28930,60000,143000.0
12439,60000,32500.0
19491,30000,61500.0
...,...,...
4946,150000,182000.0
17823,30000,51500.0
14806,340000,467000.0
20750,230000,132000.0


In [33]:
credit_subtrain["predicted"] = fit_credit_pipe.predict(credit_subtrain[col])
sq_err = (credit_subtrain["predicted"] - credit_subtrain["Credit Limit"])**2
print("Training error (RMSE):", np.sqrt(sq_err.mean()))

sq_err = (credit_val["predicted"] - credit_val["Credit Limit"])**2
print("Validation error (RMSPE):", np.sqrt(sq_err.mean()))

print("Coefficient of variation:", 1 - sq_err.mean() / credit_val["Credit Limit"].var())

Training error (RMSE): 94445.28738257817
Validation error (RMSPE): 98854.68465676498
Coefficient of variation: 0.4231791167665613


My training error is 94445 and my validation error is 98855. This shows that my model's prediction is off about 98,000 NTD, which is very large. On the other hand my coefficient of variation is about 0.4, which my model explains around 40 percent of variation in credit limits. This shows that mu model did not perfrom well and shows that the bank may use other information to predict which is not in my data. 


### 5. For Excellent: Visualize the predictions of your model. 

Using examples from class and lab, visualize the predictions of your model when plotting one of your predictors as an x-variable and the model's predictions as a y-variable. (If you have more than one predictor, pick one to use as the x-variable, and use the average value of the other one(s) as the y-variable; alternately, if one of your predictors is categorical, you can do as we did in lab and use it as a color).

In [47]:
credit_visual = pd.DataFrame({"Education": [1]*1001 + [2]*1001 +[3]*1001 +[4]*1001,
                             "avg_payment":list(np.linspace(0, 20000, 1001))*4,})

credit_visual["Age"] = credit_subtrain["Age"].mean()
credit_visual["avg_bill"] = credit_subtrain["avg_bill"].mean()
# to make a line plot of prediction using a single predictor when model has more than one predictor: 
#make a datafra. that single predictor ranging from min to max within data
#then have all the other predictors set a average (to see how one predictor affects the model ,when all other predicots are average)
credit_visual

,Education,avg_payment,Age,avg_bill
0,1,0.0,35.515056,44433.475463
1,1,20.0,35.515056,44433.475463
2,1,40.0,35.515056,44433.475463
3,1,60.0,35.515056,44433.475463
4,1,80.0,35.515056,44433.475463
...,...,...,...,...
3999,4,19920.0,35.515056,44433.475463
4000,4,19940.0,35.515056,44433.475463
4001,4,19960.0,35.515056,44433.475463
4002,4,19980.0,35.515056,44433.475463


In [48]:
credit_visual["limit_pred"] = fit_credit_pipe.predict(credit_visual)

In [49]:
credit_visual["Education"] = credit_visual["Education"].astype(str)
px.line(credit_visual, x = "avg_payment", y = "limit_pred", color = "Education")

<!doctype html>